### Step 1: Import Libraries

Import the libraries needed for data analysis: **pandas** for dataframes and **numpy** for numeric operations.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

### Step 2: Load Data

Read the dirty CSV file into a Pandas DataFrame and call `info()` to inspect column types and missing values.

In [22]:
data = pd.read_csv('dirty_cafe_sales.csv', )


data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


### Step 3: Convert Column Types

Cast each column to the appropriate dtype (strings, numerics, datetime) to prepare for cleaning.

In [3]:
data['Transaction ID'] = data['Transaction ID'].astype(str)
data['Item'] = data['Item'].astype(str)
data['Price Per Unit'] = pd.to_numeric(data['Price Per Unit'], errors='coerce')
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')
data['Total Spent'] = pd.to_numeric(data['Total Spent'], errors='coerce')
data['Transaction Date'] = pd.to_datetime(data['Transaction Date'], errors='coerce')
data['Location'] = data['Location'].astype(str)
data['Payment Method'] = data['Payment Method'].astype(str)

### Step 4: Verify Type Conversion

Call `info()` again to ensure the dtype changes took effect.

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              10000 non-null  object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    10000 non-null  object        
 6   Location          10000 non-null  object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


### Step 5: Peek at the Data

Display the first few rows to get a sense of the values.

In [5]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Step 6: Count Missing Values

Check how many null/NA entries exist before cleaning begins.

In [6]:
data.isna().sum()

Transaction ID        0
Item                  0
Quantity            479
Price Per Unit      533
Total Spent         502
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

### Step 7: Normalize Placeholders

Replace textual placeholders like 'UNKNOWN', 'nan', and 'ERROR' with actual NaN values.

In [7]:
data = data.replace(['UNKNOWN', 'nan', 'ERROR'], np.nan)

### Step 8: Recount Missing Values

See how the placeholder normalization affected the counts.

In [8]:
data.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

### Step 9: Map Items to Prices

Build a dictionary that links each menu item to its known price(s) for later imputation.

In [9]:
d = {}
for i in data['Item'].unique():
    d[i] = data['Price Per Unit'][data['Item'] == i].unique()

d


{'Coffee': array([ 2., nan]),
 'Cake': array([ 3., nan]),
 'Cookie': array([ 1., nan]),
 'Salad': array([ 5., nan]),
 'Smoothie': array([ 4., nan]),
 nan: array([], dtype=float64),
 'Sandwich': array([ 4., nan]),
 'Juice': array([ 3., nan]),
 'Tea': array([1.5, nan])}

### Step 10: Impute Item/Price Data

Use the dictionary to fill missing prices or item names based on matching rows.

In [10]:
for key, value in d.items():
    # skip entries with no price information
    if len(value) == 0:
        continue
    if not pd.isna(value[0]):
        data.loc[(data['Item'] == key) & (data['Price Per Unit'].isna()), 'Price Per Unit'] = value[0]
        data.loc[(data['Price Per Unit'] == value[0]) & (data['Item'].isna()), 'Item'] = key

data.isna().sum()

Transaction ID         0
Item                  54
Quantity             479
Price Per Unit        54
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

### Step 11: Compute Missing Numerics

Fill remaining nulls in numeric columns using arithmetic relationships between price, quantity, and total spent.

In [11]:
if data['Price Per Unit'].isna().sum() > 0:
    data['Price Per Unit'] = data['Price Per Unit'].fillna(data['Total Spent'] / data['Quantity'])
if data['Quantity'].isna().sum() > 0:
    data['Quantity'] = data['Quantity'].fillna(data['Total Spent'] / data['Price Per Unit'])
if data['Total Spent'].isna().sum() > 0:
    data['Total Spent'] = data['Total Spent'].fillna(data['Quantity'] * data['Price Per Unit'])


data.isna().sum()

Transaction ID         0
Item                  54
Quantity              23
Price Per Unit         6
Total Spent           23
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

### Step 12: Re-run Imputation

After calculating numeric values, run the item–price imputation again to catch any newly available data.

In [12]:
for key, value in d.items():
    # skip entries with no price information
    if len(value) == 0:
        continue
    if not pd.isna(value[0]):
        data.loc[(data['Item'] == key) & (data['Price Per Unit'].isna()), 'Price Per Unit'] = value[0]
        data.loc[(data['Price Per Unit'] == value[0]) & (data['Item'].isna()), 'Item'] = key

data.isna().sum()

Transaction ID         0
Item                   6
Quantity              23
Price Per Unit         6
Total Spent           23
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

### Step 13: Drop Rows with Critical Missing Data

Remove any rows that still lack an item or quantity since they can't be used for analysis.

In [16]:
data = data.dropna(subset=['Item', 'Quantity'])

### Step 14: Final Missing Value Check

Quickly verify if any missing entries remain after all cleaning steps.

In [17]:
data.isna().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      3168
Location            3952
Transaction Date     460
dtype: int64

### Step 15: Clean Categorical & Duplicates

Standardize text columns and remove duplicate transactions.

In [18]:
# clean remaining categorical columns
for col in ['Item', 'Location', 'Payment Method']:
    # strip whitespace and standardize capitalization
    data[col] = data[col].str.strip().str.title()

# fill any remaining missing categorical values with mode
for col in ['Item', 'Location', 'Payment Method']:
    if data[col].isna().any():
        mode = data[col].mode(dropna=True)
        if not mode.empty:
            data[col] = data[col].fillna(mode[0])

# drop duplicate transactions if any
data = data.drop_duplicates(subset=['Transaction ID'])

# final sanity check
data.isna().sum()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_14084\2680390643.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = data[col].str.strip().str.title()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_14084\2680390643.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = data[col].fillna(mode[0])


Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

### Step 16: Handle Transaction Dates

Drop or impute rows without a date; here we drop them to keep the time series clean.

In [20]:
# drop rows where transaction date is missing
data = data.dropna(subset=['Transaction Date'])

# recheck missing
data.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

### Step 17: Export Clean Data

Save the cleaned dataframe to disk for future analysis.

In [21]:
data.to_csv('cleaned_cafe_sales.csv', index=False)
print('Cleaned dataset saved as cleaned_cafe_sales.csv')

Cleaned dataset saved as cleaned_cafe_sales.csv
